##### **Phare Data - Migration Cross Region**
Notebook used to migrate all or selected workspaces from one capacity to another.

##### **Prerequisites**
- Tenant admin will be required to identify all capacities, workspaces and items.
- XMLA Read/Write will need to be enabled on the capacity.

In [ ]:
%pip install semantic-link-labs

In [ ]:
import sempy_labs as labs
import sempy_labs.admin as admin

In [ ]:
source_capacity = '29ff332d-22b7-4042-8bcc-919178656854'    # FR capacity
target_capacity = 'cc4a2fa9-8001-42ea-b15c-aa2f805e2c47'    # US capacity

admin.assign_workspaces_to_capacity(
    source_capacity=source_capacity,
    target_capacity=target_capacity,
    workspace='ec37be96-9a8a-4f61-b5f8-c46a3b0f878d'        # Small SM
    #workspace=None
)

##### **Large SM <= 10 Gb**

In [ ]:
import pandas as pd
df_qso=pd.DataFrame(labs.list_qso_settings(dataset='cms_semantic_model_import',workspace='7c7d3816-6ba2-45c7-be78-79a2ef2e215d'))   # Large SM <= 10 Gb
df_qso["Storage Mode"].iloc[0]

In [ ]:
labs.set_semantic_model_storage_format(dataset='cms_semantic_model_import', workspace='7c7d3816-6ba2-45c7-be78-79a2ef2e215d',storage_format='Small')

In [ ]:
import pandas as pd
import datetime, requests,time

source_capacity = ''
include_workspaces = ['']
exclude_workspaces = ['']
currentTime = datetime.datetime.now()

print (f"Script started at {currentTime}")

try:
    _workspaces=pd.DataFrame(admin.list_workspaces(source_capacity))
    for index,row in _workspaces.iterrows():
        _workspaceId=row['Id']
        _workspaceName=row['Name']
        if (len(p_only_workspaces)==0 or _workspaceName in p_only_workspaces) and _workspaceName not in p_exclude_workspaces:
            print (f" - Processing workspace : {_workspaceName} - ID : {_workspaceId} ")
            _semantic_models = pd.DataFrame(fabric.list_datasets(workspace=_workspaceName))
            if(_semantic_models.empty):
                print("-- No model indentified")
            else:
                for index,row in _semantic_models.iterrows():
                    _semanticModelId=row['Dataset ID']
                    _semanticModelName=row['Dataset Name']
                    print (f"Analyze semantic model {_semanticModelName}")
                    df_qso=pd.DataFrame(labs.list_qso_settings(dataset=_semanticModelName,workspace=_workspaceName))
                    #Get the storage mode of the semantic model if the mode is Small, there is no operation if the mode is Large, the model will be analyzed and potentially stored using the Small mode if the size is less than 10GB
                    _semanticModelStorageMode=df_qso["Storage Mode"].iloc[0]                         
                    #When the Model Storage Mode is Large, the analysis will be executed
                    if(_semanticModelStorageMode=="Large"):
                        print (f"Storage format : Large")
                        print (f"Calculate model size for storage format size update")                                 
                        _modelConnection = connect_semantic_model(workspace=_workspaceName,dataset=_semanticModelName)                            
                        df_columns = pd.DataFrame(fabric.list_columns(dataset=_semanticModelName, workspace=_workspaceName, extended=True))
                        total_size = df_columns["Total Size"].sum()
                        total_size_MB=total_size / (1024**2)
                        print(f"Estimated semantic model size: {total_size_MB} MB")
                        if total_size_MB < 10240: #10GB
                            print(f"Changing storage format to Small for semantic model {_semanticModelName} with size {total_size_MB} MB")                            
                            #Update the semantic model storage format to small
                            labs.set_semantic_model_storage_format(dataset=_semanticModelName, workspace=_workspaceName,storage_format='Small')
                            #Refresh the connection after the model storage format (required by the storage mode change)                   
                            _modelConnection = connect_semantic_model(workspace=_workspaceName,dataset=_semanticModelName)                            
                            print (f"Storage format is currently changing to Small for semantic model {_semanticModelName}")
                        else:
                            print (f"Semantic model {_semanticModelName} with size {total_size_MB} larger than 10GB, skipping")                    
                    else:
                        print (f"No need to change the storage format - already small")
                    time.sleep(2) 
except Exception as e:
    print (f"Error: {e}")

In [ ]:
source_capacity = '29ff332d-22b7-4042-8bcc-919178656854'    # FR capacity
target_capacity = 'cc4a2fa9-8001-42ea-b15c-aa2f805e2c47'    # US capacity

admin.assign_workspaces_to_capacity(
    source_capacity=source_capacity,
    target_capacity=target_capacity,
    workspace='7c7d3816-6ba2-45c7-be78-79a2ef2e215d'        # Large SM <= 10 Gb
    #workspace=None
)

In [ ]:
labs.set_semantic_model_storage_format(dataset='cms_semantic_model_import', workspace='7c7d3816-6ba2-45c7-be78-79a2ef2e215d',storage_format='Large')

In [ ]:
labs.refresh_semantic_model(dataset='cms_semantic_model_import', workspace='7c7d3816-6ba2-45c7-be78-79a2ef2e215d')

##### **Large SM > 10 Gb**

In [ ]:
df_models = admin.list_datasets()
df_models_filtered = df_models[df_models['Dataset Id']=='5cfedbe9-fdbc-4f1b-9b00-3cb152eb6a1e']
df_models_filtered

In [ ]:
labs.vertipaq_analyzer(dataset='5cfedbe9-fdbc-4f1b-9b00-3cb152eb6a1e', workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be')         # Large SM > 10 Gb

In [ ]:
import sempy_labs as labs
labs.assign_workspace_to_dataflow_storage(dataflow_storage_account='bckstore', workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be')

In [ ]:
labs.backup_semantic_model(
    dataset='5cfedbe9-fdbc-4f1b-9b00-3cb152eb6a1e',
    file_path=f'cms_semantic_model_import.abf',
    allow_overwrite=True, 
    apply_compression=True,
    workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be'
)

In [ ]:
labs.refresh_semantic_model(dataset='5cfedbe9-fdbc-4f1b-9b00-3cb152eb6a1e', refresh_type='clearValues', workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be')

In [ ]:
labs.vertipaq_analyzer(dataset='5cfedbe9-fdbc-4f1b-9b00-3cb152eb6a1e', workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be')         # Large SM > 10 Gb

In [ ]:
labs.set_semantic_model_storage_format(dataset='5cfedbe9-fdbc-4f1b-9b00-3cb152eb6a1e', workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be',storage_format='Small')

In [ ]:
source_capacity = '29ff332d-22b7-4042-8bcc-919178656854'    # FR capacity
target_capacity = 'cc4a2fa9-8001-42ea-b15c-aa2f805e2c47'    # US capacity

admin.assign_workspaces_to_capacity(
    source_capacity=source_capacity,
    target_capacity=target_capacity,
    workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be'        # Large SM <= 10 Gb
    #workspace=None
)

In [ ]:
labs.set_semantic_model_storage_format(dataset='5cfedbe9-fdbc-4f1b-9b00-3cb152eb6a1e', workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be',storage_format='Large')

In [ ]:
labs.restore_semantic_model(
    dataset='cms_semantic_model_import',
    file_path=f'cms_semantic_model_import.abf',
    allow_overwrite=True,
    ignore_incompatibilities=True,
    workspace='ef60e1c7-cb7c-4038-b44b-d6b937ece7be',
    force_restore=True
)